# GET 324 — Laboratory Exercise 10 (Mini-Project)
## Cloud Computing and AI Model Deployment for Engineering Applications

# 🧱 Concrete Bridge Deck Crack Detection (Cracked vs Non-Cracked)

**Binary Image Classification with CNN / Transfer Learning, deployed as a Streamlit web app**

---

**Group / Team Members** *(fill in before submission)*

| # | Name | Registration Number | GitHub Username | Contribution |
|---|------|---------------------|------------------|---------------|
| 1 |      |                      |                  | Dataset prep & preprocessing |
| 2 |      |                      |                  | Model development & training |
| 3 |      |                      |                  | Model evaluation |
| 4 |      |                      |                  | Streamlit app development |
| 5 |      |                      |                  | Cloud deployment |
| 6 |      |                      |                  | Documentation & report writing |

**Supervisor / Course:** GET 324 &nbsp;|&nbsp; **Assigned Task:** Binary classification — concrete bridge deck crack detection


## Course Learning Outcomes addressed in this notebook

- **CLO5** — Design, train, and evaluate a deep learning architecture (custom CNN + transfer learning) using TensorFlow/Keras for image data.
- **CLO7** — Deploy the trained model as a cloud-based web application using **Streamlit**, with the project managed on **Git/GitHub**.
- **CLO8** — Document the experimental procedure, interpret the results, and communicate the findings in a structured report.

## Project overview

Concrete bridge decks develop surface cracks over time due to fatigue, load, weathering, and material degradation. Manual visual inspection is slow, subjective, and expensive at scale. This project builds an **image classification model** that automatically labels an image of a concrete surface as **Cracked** or **Non-Cracked**, then wraps the trained model in a simple Streamlit application that can be deployed to the cloud so an inspector could upload a photo and get an instant prediction.

**Pipeline covered in this notebook:**
1. Real-world dataset acquisition (no synthetic data)
2. Exploratory data analysis
3. Preprocessing & train/validation/test split
4. Custom CNN model
5. Transfer-learning model (MobileNetV2)
6. Evaluation & model comparison
7. Saving the best model
8. Streamlit application (`app.py`)
9. `requirements.txt`, `README.md`, Git/GitHub workflow, and cloud deployment steps
10. Final mini-project report (100–150 words)


## 1. Environment setup

In [ ]:
# Run this once per Colab session.
# tensorflow, matplotlib, seaborn, scikit-learn, kagglehub are needed;
# streamlit is only needed if you want to test the app inside Colab.
!pip install -q kagglehub streamlit pyngrok --upgrade
print("Setup complete.")


In [ ]:
import os
import shutil
import pathlib
import random
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
from tensorflow.keras import layers, models, callbacks, optimizers
from sklearn.metrics import (classification_report, confusion_matrix,
                              roc_curve, auc, ConfusionMatrixDisplay)

print("TensorFlow version:", tf.__version__)
print("GPU available:", tf.config.list_physical_devices('GPU'))

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)


## 2. Dataset — real, publicly available (no synthetic data)

We use the **"Concrete Crack Images for Classification"** dataset (Özgenel & Gönenç Sorguç, 2018/2019),
originally published on **Mendeley Data** (DOI: `10.17632/5y9wdsg2zt.2`) and mirrored on Kaggle as
**`arunrk7/surface-crack-detection`**. It contains **40,000 real photographs** of concrete surfaces from
METU campus buildings — **20,000 "Positive" (cracked)** and **20,000 "Negative" (non-cracked)** images,
227×227 px, RGB. This is exactly the kind of surface-crack imagery relevant to bridge-deck inspection.

> Citation: Özgenel, Ç.F. (2019), "Concrete Crack Images for Classification", Mendeley Data, V2,
> doi: 10.17632/5y9wdsg2zt.2. Licensed CC BY 4.0.

### Option A — Download via KaggleHub (recommended, fastest in Colab)
You need a free Kaggle account and API token (`kaggle.json`).
1. Go to kaggle.com → Account → **Create New API Token** (downloads `kaggle.json`).
2. Run the cell below and upload `kaggle.json` when prompted.


In [ ]:
from google.colab import files

# Upload your kaggle.json (skip this cell if it's already configured)
if not os.path.exists("/root/.kaggle/kaggle.json"):
    print("Please upload your kaggle.json API token file:")
    uploaded = files.upload()
    os.makedirs("/root/.kaggle", exist_ok=True)
    for fname in uploaded:
        shutil.move(fname, "/root/.kaggle/kaggle.json")
    os.chmod("/root/.kaggle/kaggle.json", 0o600)
else:
    print("kaggle.json already configured.")


In [ ]:
import kagglehub

# Downloads the dataset to a local cache dir and returns the path
dataset_path = kagglehub.dataset_download("arunrk7/surface-crack-detection")
print("Dataset downloaded to:", dataset_path)

# Inspect folder structure
for root, dirs, filenames in os.walk(dataset_path):
    level = root.replace(dataset_path, "").count(os.sep)
    indent = "  " * level
    print(f"{indent}{os.path.basename(root)}/")
    if level >= 2:
        continue


### Option B — Manual download (if you don't want to use the Kaggle API)

1. Download the dataset directly from Mendeley: https://data.mendeley.com/datasets/5y9wdsg2zt/2
   (or from Kaggle: https://www.kaggle.com/datasets/arunrk7/surface-crack-detection).
2. Upload the extracted archive to your Google Drive, e.g. `MyDrive/concrete_crack_dataset.zip`.
3. Run the cell below instead of Option A.


In [ ]:
# --- Option B: uncomment and run this instead of Option A if you downloaded manually ---

# from google.colab import drive
# drive.mount('/content/drive')
# zip_path = "/content/drive/MyDrive/concrete_crack_dataset.zip"
# extract_path = "/content/concrete_crack_dataset"
# os.makedirs(extract_path, exist_ok=True)
# import zipfile
# with zipfile.ZipFile(zip_path, 'r') as zf:
#     zf.extractall(extract_path)
# dataset_path = extract_path
# print("Extracted to:", dataset_path)


In [ ]:
# Locate the two class folders regardless of how the archive is nested
def find_class_dirs(base_path, class_names=("Positive", "Negative")):
    found = {}
    for root, dirs, _ in os.walk(base_path):
        for d in dirs:
            if d in class_names:
                found[d] = os.path.join(root, d)
    return found

class_dirs = find_class_dirs(dataset_path)
print(class_dirs)

# The dataset root that CONTAINS both class folders (needed by image_dataset_from_directory)
DATA_DIR = os.path.commonpath(list(class_dirs.values()))
print("\nData directory used for loading:", DATA_DIR)

for cls, path in class_dirs.items():
    n = len(list(pathlib.Path(path).glob("*.jpg"))) + len(list(pathlib.Path(path).glob("*.png")))
    print(f"{cls}: {n} images")


## 3. Exploratory data analysis

In [ ]:
# Class balance bar chart
counts = {cls: len(os.listdir(path)) for cls, path in class_dirs.items()}
plt.figure(figsize=(5, 4))
sns.barplot(x=list(counts.keys()), y=list(counts.values()), palette="viridis")
plt.title("Class distribution")
plt.ylabel("Number of images")
plt.show()
print(counts)


In [ ]:
# Sample images from each class
fig, axes = plt.subplots(2, 5, figsize=(15, 6))
for row, (cls, path) in enumerate(class_dirs.items()):
    sample_files = random.sample(os.listdir(path), 5)
    for col, fname in enumerate(sample_files):
        img = tf.keras.utils.load_img(os.path.join(path, fname))
        axes[row, col].imshow(img)
        axes[row, col].set_title(cls)
        axes[row, col].axis("off")
plt.tight_layout()
plt.show()


## 4. Preprocessing & train / validation / test split

Images are resized to **128×128** (down from the original 227×227) to keep training fast on a
Colab GPU without losing the texture detail that distinguishes a crack from a blemish or stain.
We create an 70% / 15% / 15% train / validation / test split using `image_dataset_from_directory`.


In [ ]:
IMG_SIZE = (128, 128)
BATCH_SIZE = 32

# 85% train+val / 15% test split first
full_train_val_ds = tf.keras.utils.image_dataset_from_directory(
    DATA_DIR,
    labels="inferred",
    label_mode="binary",
    class_names=["Negative", "Positive"],   # Negative -> 0 (non-cracked), Positive -> 1 (cracked)
    validation_split=0.15,
    subset="training",
    seed=SEED,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
)

test_ds = tf.keras.utils.image_dataset_from_directory(
    DATA_DIR,
    labels="inferred",
    label_mode="binary",
    class_names=["Negative", "Positive"],
    validation_split=0.15,
    subset="validation",
    seed=SEED,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
)

class_names = full_train_val_ds.class_names
print("Classes:", class_names, " (0 = Negative/non-cracked, 1 = Positive/cracked)")

# Split the remaining 85% into ~82% train / ~18% val (=> overall ~70/15/15)
val_batches = int(0.18 * tf.data.experimental.cardinality(full_train_val_ds).numpy())
val_ds = full_train_val_ds.take(val_batches)
train_ds = full_train_val_ds.skip(val_batches)

print("Train batches:", tf.data.experimental.cardinality(train_ds).numpy())
print("Val batches:  ", tf.data.experimental.cardinality(val_ds).numpy())
print("Test batches: ", tf.data.experimental.cardinality(test_ds).numpy())


In [ ]:
AUTOTUNE = tf.data.AUTOTUNE

# Cache + prefetch for speed
train_ds = train_ds.cache().shuffle(1000, seed=SEED).prefetch(buffer_size=AUTOTUNE)
val_ds = val_ds.cache().prefetch(buffer_size=AUTOTUNE)
test_ds_eval = test_ds.cache().prefetch(buffer_size=AUTOTUNE)

# Data augmentation (applied only during training, inside the model graph)
data_augmentation = tf.keras.Sequential([
    layers.RandomFlip("horizontal_and_vertical"),
    layers.RandomRotation(0.05),
    layers.RandomZoom(0.1),
    layers.RandomContrast(0.1),
], name="data_augmentation")

rescale = layers.Rescaling(1.0 / 255)


## 5. Model 1 — Custom CNN (baseline)

A compact CNN built from scratch: three convolutional blocks (Conv2D + BatchNorm + MaxPool),
followed by a dense classification head with dropout for regularisation.


In [ ]:
def build_custom_cnn(input_shape=(128, 128, 3)):
    inputs = tf.keras.Input(shape=input_shape)
    x = data_augmentation(inputs)
    x = rescale(x)

    for filters in [32, 64, 128]:
        x = layers.Conv2D(filters, 3, padding="same", activation="relu")(x)
        x = layers.BatchNormalization()(x)
        x = layers.MaxPooling2D()(x)

    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dense(128, activation="relu")(x)
    x = layers.Dropout(0.4)(x)
    outputs = layers.Dense(1, activation="sigmoid")(x)

    return tf.keras.Model(inputs, outputs, name="custom_cnn")

custom_cnn = build_custom_cnn()
custom_cnn.compile(
    optimizer=optimizers.Adam(learning_rate=1e-3),
    loss="binary_crossentropy",
    metrics=["accuracy", tf.keras.metrics.Precision(name="precision"),
             tf.keras.metrics.Recall(name="recall")],
)
custom_cnn.summary()


In [ ]:
EPOCHS = 12

cnn_callbacks = [
    callbacks.EarlyStopping(monitor="val_loss", patience=3, restore_best_weights=True),
    callbacks.ModelCheckpoint("best_custom_cnn.keras", monitor="val_accuracy",
                               save_best_only=True),
    callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=2),
]

history_cnn = custom_cnn.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS,
    callbacks=cnn_callbacks,
)


In [ ]:
def plot_history(history, title):
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    axes[0].plot(history.history["accuracy"], label="train")
    axes[0].plot(history.history["val_accuracy"], label="val")
    axes[0].set_title(f"{title} — Accuracy")
    axes[0].set_xlabel("Epoch"); axes[0].legend()

    axes[1].plot(history.history["loss"], label="train")
    axes[1].plot(history.history["val_loss"], label="val")
    axes[1].set_title(f"{title} — Loss")
    axes[1].set_xlabel("Epoch"); axes[1].legend()
    plt.tight_layout()
    plt.show()

plot_history(history_cnn, "Custom CNN")


## 6. Model 2 — Transfer learning (MobileNetV2)

MobileNetV2 pretrained on ImageNet is used as a frozen feature extractor, with a new classification
head trained on top. MobileNetV2 is small and fast enough to later fine-tune and deploy on a free
Streamlit Cloud instance.


In [ ]:
def build_transfer_model(input_shape=(128, 128, 3)):
    base_model = tf.keras.applications.MobileNetV2(
        input_shape=input_shape, include_top=False, weights="imagenet"
    )
    base_model.trainable = False  # freeze for the first training phase

    inputs = tf.keras.Input(shape=input_shape)
    x = data_augmentation(inputs)
    x = tf.keras.applications.mobilenet_v2.preprocess_input(x)
    x = base_model(x, training=False)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dropout(0.3)(x)
    outputs = layers.Dense(1, activation="sigmoid")(x)

    model = tf.keras.Model(inputs, outputs, name="mobilenetv2_transfer")
    return model, base_model

transfer_model, base_model = build_transfer_model()
transfer_model.compile(
    optimizer=optimizers.Adam(learning_rate=1e-3),
    loss="binary_crossentropy",
    metrics=["accuracy", tf.keras.metrics.Precision(name="precision"),
             tf.keras.metrics.Recall(name="recall")],
)
transfer_model.summary()


In [ ]:
transfer_callbacks = [
    callbacks.EarlyStopping(monitor="val_loss", patience=3, restore_best_weights=True),
    callbacks.ModelCheckpoint("best_transfer_model.keras", monitor="val_accuracy",
                               save_best_only=True),
]

history_transfer = transfer_model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS,
    callbacks=transfer_callbacks,
)


In [ ]:
# Optional fine-tuning phase: unfreeze the top layers of MobileNetV2 and train with a low LR
base_model.trainable = True
fine_tune_at = len(base_model.layers) - 30
for layer in base_model.layers[:fine_tune_at]:
    layer.trainable = False

transfer_model.compile(
    optimizer=optimizers.Adam(learning_rate=1e-5),
    loss="binary_crossentropy",
    metrics=["accuracy", tf.keras.metrics.Precision(name="precision"),
             tf.keras.metrics.Recall(name="recall")],
)

fine_tune_epochs = 5
history_fine = transfer_model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=fine_tune_epochs,
    callbacks=transfer_callbacks,
)


In [ ]:
plot_history(history_transfer, "MobileNetV2 (frozen base)")
plot_history(history_fine, "MobileNetV2 (fine-tuned)")


## 7. Evaluation & model comparison on the held-out test set

In [ ]:
def evaluate_model(model, ds, name):
    y_true, y_pred_probs = [], []
    for images, labels_batch in ds:
        preds = model.predict(images, verbose=0)
        y_true.extend(labels_batch.numpy().flatten().tolist())
        y_pred_probs.extend(preds.flatten().tolist())

    y_true = np.array(y_true)
    y_pred_probs = np.array(y_pred_probs)
    y_pred = (y_pred_probs >= 0.5).astype(int)

    print(f"\n===== {name} — Test set performance =====")
    print(classification_report(y_true, y_pred, target_names=["Non-cracked", "Cracked"]))

    cm = confusion_matrix(y_true, y_pred)
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=["Non-cracked", "Cracked"])
    disp.plot(cmap="Blues")
    plt.title(f"{name} — Confusion Matrix")
    plt.show()

    fpr, tpr, _ = roc_curve(y_true, y_pred_probs)
    roc_auc = auc(fpr, tpr)
    plt.figure(figsize=(5, 4))
    plt.plot(fpr, tpr, label=f"AUC = {roc_auc:.3f}")
    plt.plot([0, 1], [0, 1], "k--")
    plt.xlabel("False Positive Rate"); plt.ylabel("True Positive Rate")
    plt.title(f"{name} — ROC Curve")
    plt.legend()
    plt.show()

    test_loss, test_acc, test_prec, test_rec = model.evaluate(ds, verbose=0)
    return {"model": name, "accuracy": test_acc, "precision": test_prec,
            "recall": test_rec, "auc": roc_auc}

results = []
results.append(evaluate_model(custom_cnn, test_ds_eval, "Custom CNN"))
results.append(evaluate_model(transfer_model, test_ds_eval, "MobileNetV2 (fine-tuned)"))

import pandas as pd
results_df = pd.DataFrame(results)
results_df


In [ ]:
# Pick the best-performing model (by test accuracy) as the one we ship to the app
best_row = results_df.loc[results_df["accuracy"].idxmax()]
print("Best model:", best_row["model"])

best_model = transfer_model if best_row["model"] == "MobileNetV2 (fine-tuned)" else custom_cnn


## 8. Save the trained model for deployment

We save in the native Keras format (`.keras`) used by the Streamlit app, and also export a
`SavedModel` folder as a portable backup.


In [ ]:
os.makedirs("model", exist_ok=True)
best_model.save("model/crack_detector.keras")
best_model.export("model/crack_detector_savedmodel")

print("Saved:")
print(" - model/crack_detector.keras")
print(" - model/crack_detector_savedmodel/")

# Download to your local machine (for adding to the GitHub repo)
from google.colab import files as colab_files
colab_files.download("model/crack_detector.keras")


## 9. Streamlit application (`app.py`)

This is the complete source code for the deployable Streamlit web app (CLO7 deliverable #1).
Running the cell below writes `app.py` to disk inside this Colab environment. Copy this file into
your GitHub repository exactly as-is.


In [ ]:
%%writefile app.py
"""
Concrete Bridge Deck Crack Detection - Streamlit App
GET 324 Mini-Project (Laboratory Exercise 10)

Run locally with:  streamlit run app.py
"""

import numpy as np
import streamlit as st
import tensorflow as tf
from PIL import Image

st.set_page_config(
    page_title="Concrete Crack Detector",
    page_icon="\U0001F309",
    layout="centered",
)

IMG_SIZE = (128, 128)
MODEL_PATH = "model/crack_detector.keras"
CLASS_NAMES = ["Non-cracked", "Cracked"]


@st.cache_resource
def load_model():
    return tf.keras.models.load_model(MODEL_PATH)


def preprocess_image(pil_image: Image.Image) -> np.ndarray:
    img = pil_image.convert("RGB").resize(IMG_SIZE)
    arr = tf.keras.utils.img_to_array(img)
    arr = np.expand_dims(arr, axis=0)
    return arr


def predict(model, pil_image: Image.Image):
    arr = preprocess_image(pil_image)
    prob_cracked = float(model.predict(arr, verbose=0)[0][0])
    label = CLASS_NAMES[1] if prob_cracked >= 0.5 else CLASS_NAMES[0]
    confidence = prob_cracked if prob_cracked >= 0.5 else 1 - prob_cracked
    return label, confidence, prob_cracked


def main():
    st.title("\U0001F309 Concrete Bridge Deck Crack Detector")
    st.write(
        "Upload a photo of a concrete surface (e.g. a bridge deck) and the model "
        "will classify it as **Cracked** or **Non-cracked**."
    )

    with st.sidebar:
        st.header("About")
        st.markdown(
            "- **Course:** GET 324 — Laboratory Exercise 10\n"
            "- **Task:** Binary image classification (cracked vs non-cracked)\n"
            "- **Model:** CNN / MobileNetV2 transfer learning\n"
            "- **Dataset:** Concrete Crack Images for Classification "
            "(Ozgenel & Gonenc Sorguc, Mendeley Data)"
        )

    model = load_model()

    uploaded_file = st.file_uploader(
        "Choose an image...", type=["jpg", "jpeg", "png"]
    )

    col1, col2 = st.columns(2)

    if uploaded_file is not None:
        image = Image.open(uploaded_file)
        with col1:
            st.image(image, caption="Uploaded image", use_container_width=True)

        with st.spinner("Analysing image..."):
            label, confidence, prob_cracked = predict(model, image)

        with col2:
            if label == "Cracked":
                st.error(f"### \U0001F6A8 {label}")
            else:
                st.success(f"### \u2705 {label}")
            st.metric("Confidence", f"{confidence * 100:.1f}%")
            st.progress(prob_cracked)
            st.caption(f"Raw model output (P(cracked) = {prob_cracked:.3f})")

        st.info(
            "\u26A0\uFE0F This tool supports visual inspection but does not replace "
            "professional structural assessment of bridge decks."
        )
    else:
        st.write("\u2b06\ufe0f Upload an image to get a prediction.")


if __name__ == "__main__":
    main()


In [ ]:
%%writefile requirements.txt
streamlit>=1.32
tensorflow-cpu>=2.15
numpy>=1.24
Pillow>=10.0


## 10. Testing the app inside Colab (optional, quick sanity check)

Streamlit Cloud is the real deployment target (step 11), but you can smoke-test the app right
here in Colab using `pyngrok` before pushing to GitHub.


In [ ]:
# Optional local smoke test using a tunnel (requires a free ngrok authtoken:
# https://dashboard.ngrok.com/get-started/your-authtoken)
#
# from pyngrok import ngrok
# ngrok.set_auth_token("YOUR_NGROK_TOKEN")
# public_url = ngrok.connect(8501)
# print("Streamlit app will be available at:", public_url)
# get_ipython().system_raw('streamlit run app.py &')


## 11. Git, GitHub, and Streamlit Cloud deployment (CLO7)

Run these commands **on your local machine or in a terminal** (not required to execute inside
Colab) after downloading `app.py`, `requirements.txt`, the `model/` folder, and this notebook.

### 11.1 Recommended repository structure
```
concrete-crack-detector/
├── app.py
├── requirements.txt
├── README.md
├── model/
│   └── crack_detector.keras
└── notebooks/
    └── GET324_Crack_Detection_MiniProject.ipynb
```

### 11.2 Initialise Git and push to GitHub
```bash
git init
git add .
git commit -m "Initial commit: concrete crack detection app"
git branch -M main
git remote add origin https://github.com/<your-username>/concrete-crack-detector.git
git push -u origin main
```

### 11.3 Deploy on Streamlit Community Cloud
1. Go to https://share.streamlit.io and sign in with GitHub.
2. Click **New app**, choose your repository, branch `main`, and main file path `app.py`.
3. Click **Deploy**. Streamlit installs `requirements.txt` and starts the app automatically.
4. Copy the generated public URL (e.g. `https://concrete-crack-detector.streamlit.app`) — this is
   your deliverable #3.

> If the model file is too large for GitHub's normal limits, use **Git LFS**
> (`git lfs track "*.keras"`) or host the model on Google Drive / Hugging Face and download it
> inside `app.py` at startup with `gdown` or `huggingface_hub`.


## 12. `README.md` (deliverable #2 — included in the GitHub repo)

Run the cell below to generate the README file exactly as it should appear in the repository.


In [ ]:
%%writefile README.md
# Concrete Bridge Deck Crack Detection

Binary image classification app (Cracked vs Non-cracked) built for **GET 324 — Laboratory
Exercise 10 (Mini-Project): Cloud Computing and AI Model Deployment for Engineering Applications**.

## Problem statement
Bridge decks and other concrete structures develop surface cracks over time. This project trains
a Convolutional Neural Network to automatically classify an image of a concrete surface as
**Cracked** or **Non-cracked**, then deploys the model as a Streamlit web application.

## Dataset
"Concrete Crack Images for Classification" — Ozgenel, C.F. & Gonenc Sorguc, A. (2018),
Mendeley Data, V2, DOI: 10.17632/5y9wdsg2zt.2 (also mirrored on Kaggle as
`arunrk7/surface-crack-detection`). 40,000 real photographs (227x227 px) of concrete surfaces
from METU campus buildings, evenly split between cracked and non-cracked classes. No synthetic
data was used.

## Models trained
1. **Custom CNN** — 3 convolutional blocks with batch normalisation and max-pooling, trained from
   scratch.
2. **MobileNetV2 (transfer learning)** — ImageNet-pretrained backbone, frozen then fine-tuned.

Both models are trained, evaluated, and compared inside
`notebooks/GET324_Crack_Detection_MiniProject.ipynb` using accuracy, precision, recall, a
confusion matrix, and ROC-AUC on a held-out test split. The better-performing model is exported
to `model/crack_detector.keras` and used by the app.

## Repository structure
```
concrete-crack-detector/
├── app.py                  # Streamlit application
├── requirements.txt        # Python dependencies
├── README.md
├── model/
│   └── crack_detector.keras
└── notebooks/
    └── GET324_Crack_Detection_MiniProject.ipynb
```

## Running locally
```bash
git clone https://github.com/<your-username>/concrete-crack-detector.git
cd concrete-crack-detector
pip install -r requirements.txt
streamlit run app.py
```
Then open the local URL Streamlit prints (usually http://localhost:8501).

## Using the app
1. Open the deployed app URL (or the local URL above).
2. Upload a photo of a concrete surface (`.jpg`, `.jpeg`, or `.png`).
3. The app displays the predicted class (**Cracked** / **Non-cracked**) and a confidence score.

## Deployment
Deployed on **Streamlit Community Cloud**: `https://<your-app-name>.streamlit.app`
(replace with your actual deployed URL before submission).

## Team
See the notebook header for the full list of team members, registration numbers, and individual
contributions.

## Citation
Ozgenel, C.F. (2019), "Concrete Crack Images for Classification", Mendeley Data, V2,
doi: 10.17632/5y9wdsg2zt.2. Licensed under CC BY 4.0.


## 13. Project report (100–150 words) — deliverable #4

*(Copy the paragraph below into your submission document, or leave it here as part of the
notebook.)*

> This project addresses binary classification of concrete surfaces into **cracked** and
> **non-cracked** categories, relevant to bridge-deck inspection. We used the real-world
> "Concrete Crack Images for Classification" dataset (Özgenel & Gönenç Sorguç, Mendeley Data,
> 40,000 images), split 70/15/15 into train, validation, and test sets. A custom CNN and a
> fine-tuned MobileNetV2 transfer-learning model were trained and compared using accuracy,
> precision, recall, and ROC-AUC; the stronger model was exported and served through a Streamlit
> web application that lets a user upload an image and receive an instant crack/no-crack
> prediction with a confidence score. The main challenges were dataset size (slow epochs on
> free-tier GPUs) and keeping the deployed model small enough for Streamlit Cloud; these were
> solved by downsizing images to 128×128, using `EarlyStopping`, and choosing a lightweight
> MobileNetV2 backbone. Future improvements include Grad-CAM visualisation of crack regions and
> testing on external bridge-deck photos to check real-world generalisation.


## 14. Assessment checklist (per assignment brief)

- [x] Dataset preparation and preprocessing — real, public dataset, 70/15/15 split, augmentation
- [x] Model development and training — custom CNN **and** transfer learning (MobileNetV2)
- [x] Model evaluation — accuracy, precision, recall, confusion matrix, ROC-AUC, model comparison
- [x] Application development — `app.py` Streamlit app with upload + prediction UI
- [x] Cloud deployment — step-by-step Streamlit Community Cloud instructions (§11)
- [x] Documentation and report — `README.md` (§12) and 100–150 word report (§13)
- [ ] Fill in team members table (top of notebook), push to GitHub, and paste the live deployed
      app URL into the README before submission.
